# Iraqi Voice Studio — تدريب RVC v2 على Google Colab

دفتر Colab الرسمي لمنصة **استوديو الصوت العراقي**.

يُنفّذ عبر سكربت التثبيت الرسمي `scripts/colab/install_colab.sh` (يتعامل تلقائيًا مع GPU/CUDA/uv/Drive/الاستكمال).

المتغيرات القابلة للتعديل في الخلية الأولى:
- `IRAQI_VOICE_REPO`: مستودع git للمشروع (افتراضيًا المستودع العام؛ اضبطه لنسختك الخاصة).
- `DRIVE_DIR`: مجلد Drive الذي تُحفظ فيه النماذج ونقاط التحقق (افتراضي `RVC_Iraqi`).
- ضع ملفات البيانات الصوتية العراقية (10–60 دقيقة، نظيفة) في `Drive/MyDrive/<DRIVE_DIR>/datasets/<اسم>/`.

**ترتيب التشغيل:** 1 → 2 → 3 → 4، و5 اختيارية. شغّل الخلية الأخيرة (Snapshot) قبل انتهاء الجلسة أو عند أي انقطاع.

In [ ]:
# @title 1) الإعداد والتحقق من البيئة (GPU / CUDA / Drive)
IRAQI_VOICE_REPO = ""  # @param {type:"string"}
DRIVE_DIR = "RVC_Iraqi"  # @param {type:"string"}

import os
if IRAQI_VOICE_REPO:
    os.environ["IRAQI_VOICE_REPO"] = IRAQI_VOICE_REPO
os.environ["DRIVE_DIR"] = DRIVE_DIR

!nvidia-smi -L || echo "NO GPU"
!python3 --version
print("الخلية التالية: التثبيت الرسمي ثم ربط Drive.")

In [ ]:
# @title 2) التثبيت الرسمي (uv + بيئة + أصول مُسبقة + ربط Drive)
# ينسخ المستودع إلى /content/iraqi-voice-platform ثم يثبّت كل شيء.
import os
if os.environ.get("IRAQI_VOICE_REPO"):
    !rm -rf /content/iraqi-voice-platform
    !git clone --depth 1 "$IRAQI_VOICE_REPO" /content/iraqi-voice-platform
%cd /content/iraqi-voice-platform

!bash scripts/colab/install_colab.sh

# تفعيل توجيه Drive داخل نواة الجلسة — كل أمر !uv run لاحق يكتب
# checkpoints/exports مباشرة إلى Drive (لا حاجة لأي source في الخلايا التالية).
if os.path.exists("colab_wire_env.py"):
    exec(open("colab_wire_env.py").read())
    print(">>> توجيه Drive نشط داخل الجلسة <<<")
else:
    print("تعذّر ربط Drive — سيُخزَّن كل شيء محليًا (غير دائم).")

In [ ]:
# @title 3) تجهيز البيانات العراقية (تحقق → قص → F0 → ميزات → فهرس)
EXPERIMENT = "iraqi_voice"  # @param {type:"string"}
DATASET_IN_DRIVE = "RVC_Iraqi/datasets/my_voice"  # @param {type:"string"}
SR = "40k"  # @param ["32k", "40k", "48k"]
F0_METHOD = "rmvpe"  # @param ["rmvpe", "fcpe", "pm"]

# متغيرات الخلية 2 (توجيه Drive) موروثة تلقائيًا هنا لأنها في نفس النواة
!uv run rvc-platform data-check "/content/drive/MyDrive/$DATASET_IN_DRIVE"
!uv run rvc-platform prepare-dataset "/content/drive/MyDrive/$DATASET_IN_DRIVE" "$EXPERIMENT" --sr "$SR" --f0-method "$F0_METHOD"

In [ ]:
# @title 4) التدريب (يُستكمل تلقائيًا من آخر Checkpoint عند إعادة التشغيل)
EPOCHS = 200  # @param {type:"integer"}
BATCH = 6  # @param {type:"integer"}
RESUME_FROM = ""  # @param {type:"string"} (اختياري: اسم نقطة تحقق مثل G_48.pth أو latest — اتركه فارغًا لاستكمال الأحدث تلقائيًا)

cmd = f"uv run rvc-platform train start --experiment $EXPERIMENT --epochs {EPOCHS} --batch {BATCH} --sr $SR"
if RESUME_FROM:
    cmd += f" --resume-from {RESUME_FROM}"
print(">>>", cmd)
!{cmd}

# إيقاف تعاوني (يحفظ checkpoint قبل الخروج) من نافذة أخرى أو خلية منفصلة:
#   !uv run rvc-platform train stop

In [ ]:
# @title 5) (اختياري) واجهة الويب عبر نفق Cloudflare مؤقت
# ملاحظة: النفق يُفعّل المصادقة تلقائيًا — الرمز في logs/auth.token
!uv run rvc-platform web --host 0.0.0.0 --tunnel

In [ ]:
# @title 6) Snapshot إلى Drive قبل انتهاء الجلسة (شغّلها يدويًا عند الانقطاع)
# يوقف التدريب تعاونيًا (مع حفظ checkpoint) ثم ينسخ runs/ و models/ إلى Drive.
!uv run rvc-platform train stop
!uv run rvc-platform colab snapshot
!uv run rvc-platform colab status
print(">>> حالّت الجلسة: البيانات محفوظة في Drive. عند جلسة جديدة: شغّل الخليتين 2 و4 فقط — سيُستكمل التدريب تلقائيًا. <<<")